# 05_Dashboard_Preparation: Export Dashboard Tables

This notebook prepares dashboard-ready datasets from the cleaned transactions, enabling interactive visualisations in Power BI.


## Setup and Load Data

Load the cleaned dataset and prepare tables for dashboard reporting.

In [ ]:

from pathlib import Path
import pandas as pd

DATA_PROCESSED_DIR = Path('data/processed')
DASHBOARD_DATA_DIR = Path('dashboards/dashboard_data')
DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED_DIR / 'cleaned_transactions.csv')


## KPI Summary Table

Compute key metrics such as total transactions, number of suspicious transactions, and average amount.

In [ ]:

kpi_summary = {
    'total_transactions': len(df),
    'suspicious_transactions': int(df['is_laundering'].sum()) if 'is_laundering' in df.columns else None,
    'avg_amount': df['amount'].mean() if 'amount' in df.columns else None
}

kpi_df = pd.DataFrame([kpi_summary])
kpi_df.to_csv(DASHBOARD_DATA_DIR / 'kpi_summary.csv', index=False)
print('Saved KPI summary.')


## Transaction Trend Table

Aggregate transactions over time to analyze trends, if a datetime column exists.

In [ ]:

if 'datetime' in df.columns:
    df['date'] = pd.to_datetime(df['datetime']).dt.date
    trend_df = df.groupby('date').agg(
        total_transactions=('date', 'size'),
        suspicious_transactions=('is_laundering', 'sum') if 'is_laundering' in df.columns else ('date','size'),
        total_amount=('amount', 'sum') if 'amount' in df.columns else ('date','size')
    ).reset_index()
    trend_df.to_csv(DASHBOARD_DATA_DIR / 'transaction_trend.csv', index=False)
    print('Saved transaction trend table.')


## Suspicious Transactions by Payment Type and Currency

Summarize suspicious activity by payment type and currency for dashboard visuals.

In [ ]:

if {'payment_type','is_laundering'}.issubset(df.columns):
    suspicious_by_payment = df[df['is_laundering']==1].groupby('payment_type').size().reset_index(name='count')
    suspicious_by_payment.to_csv(DASHBOARD_DATA_DIR / 'suspicious_by_payment_type.csv', index=False)
if {'currency','is_laundering'}.issubset(df.columns):
    suspicious_by_currency = df[df['is_laundering']==1].groupby('currency').size().reset_index(name='count')
    suspicious_by_currency.to_csv(DASHBOARD_DATA_DIR / 'suspicious_by_currency.csv', index=False)
    print('Saved suspicious transactions by payment type and currency tables.')


## High-Risk Accounts Table

Identify and export high-risk accounts based on the risk bands computed earlier.

In [ ]:

if {'sender_account','is_laundering'}.issubset(df.columns):
    sender_rate = df.groupby('sender_account')['is_laundering'].mean()
    high_risk_accounts = sender_rate[sender_rate > sender_rate.quantile(0.66)].reset_index()
    high_risk_accounts.to_csv(DASHBOARD_DATA_DIR / 'high_risk_accounts.csv', index=False)
    print('Saved high-risk accounts table.')


## Model Prediction Summary (Optional)

If model predictions are available, load them and export aggregated metrics per account.

In [ ]:

# Example: load predictions and summarise by account
try:
    from pathlib import Path
    import joblib
    model = joblib.load('models/trained_model.joblib')
    # Suppose features X have been prepared similarly as in model training
    # preds = model.predict_proba(X)[:,1]
    # df['pred_prob'] = preds
    # summary = df.groupby('sender_account')['pred_prob'].mean().reset_index()
    # summary.to_csv(DASHBOARD_DATA_DIR / 'model_prediction_summary.csv', index=False)
    print('Model prediction summary would be generated here if predictions are available.')
except Exception as e:
    print('Model not available or predictions could not be computed:', e)


## Business Interpretation

Explain how the exported tables support Power BI dashboard creation.

In [ ]:

print('The generated tables provide a structured input for interactive dashboards:')
print('- KPI summary informs top-level metrics.')
print('- Trend tables allow time series analysis of suspicious activity.')
print('- Payment type and currency tables highlight typology-specific risks.')
print('- High-risk account list helps focus investigations.')
